# Bulgarian mT5 All-in-One Corrector

This notebook trains a single Bulgarian text-correction model with **mT5** that aims to fix:

1. spelling mistakes
2. punctuation mistakes
3. grammar mistakes

It is designed for one-shot training and easy comparison against your pipeline.


In [10]:

import os
import re
import json
import math
import random
import string
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


## 2) Configuration

In [11]:


def detect_data_root() -> str:
    for candidate in [
        "/workspace",
        ".",
    ]:
        if os.path.exists(candidate):
            return candidate
    return "."

DATA_ROOT = detect_data_root()
OUTPUT_DIR = "./mt5_bg_full_corrector"

MODEL_NAME = "google/mt5-base"

MAX_INPUT_LENGTH = 160
MAX_TARGET_LENGTH = 160

NOISY_VARIANTS_PER_SENTENCE = 4
MAX_SENTENCES = 35000
IDENTITY_RATIO = 0.20

TEST_SIZE = 0.10
VALIDATION_SIZE = 0.10  

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
USE_FP16 = torch.cuda.is_available() and not USE_BF16

print("DATA_ROOT:", DATA_ROOT)
print("BF16:", USE_BF16)
print("FP16:", USE_FP16)


DATA_ROOT: /workspace
BF16: True
FP16: False


## 3) Load the dataset

In [20]:

def list_data_files(root: str) -> List[str]:
    found = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith((".csv", ".tsv", ".txt", ".json", ".jsonl")):
                found.append(os.path.join(dirpath, fn))
    return sorted(found)

data_files = list_data_files(DATA_ROOT)
print("Found files:")
for f in data_files:
    print(" -", f)

def read_tabular_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".tsv":
        return pd.read_csv(path, sep="\t")
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".jsonl":
        return pd.read_json(path, lines=True)
    if ext == ".json":
        return pd.read_json(path)
    if ext == ".txt":
        # one sentence per line fallback
        return pd.DataFrame({"text": open(path, "r", encoding="utf-8").read().splitlines()})
    raise ValueError(f"Unsupported file type: {path}")

for f in data_files[:5]:
    try:
        df = read_tabular_file(f)
        print("\nFILE:", os.path.basename(f))
        print("Columns:", list(df.columns))
        print(df.head(3))
    except Exception as e:
        print("Could not preview", f, "->", e)

Found files:
 - /workspace/.cache/huggingface/.agent_harnesses.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/.no_exist/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/added_tokens.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/.no_exist/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/model.safetensors.index.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/.no_exist/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/tokenizer.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/snapshots/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/config.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/snapshots/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/generation_config.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/snapshots/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/special_tokens_map.json
 - /workspace/.cache/huggingface/hub/models--google--mt5-base/snapshots/2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f/tokenizer_config.jso

In [25]:

FILE_HINT = "sents.tsv"
TEXT_COLUMN = "sentence"

chosen_file = None
chosen_df = None
chosen_col = None

possible_paths = [
    "/workspace/sents.tsv",
    "sents.tsv",
    "./sents.tsv",
    "/workspace/data/sents.tsv",
]

for path in possible_paths:
    if os.path.exists(path):
        chosen_file = path
        break

if chosen_file is None:
    data_files = list_data_files(DATA_ROOT)
    for f in data_files:
        if "sents.tsv" in f.lower():
            chosen_file = f
            break

if chosen_file is None:
    raise RuntimeError(f"sents.tsv not found. Current DATA_ROOT={DATA_ROOT}. Run the file listing cell again.")

print(f"Found file at: {chosen_file}")

try:
    temp = read_tabular_file(chosen_file)
    print("Columns found:", list(temp.columns))
    
    if TEXT_COLUMN in temp.columns:
        chosen_col = TEXT_COLUMN
    else:
        chosen_col = find_best_text_column(temp)
    
    chosen_df = temp
    print(f"Using column: '{chosen_col}'")
    
except Exception as e:
    raise RuntimeError(f"Failed to read file: {e}")

raw_texts = (
    chosen_df[chosen_col]
    .dropna()
    .astype(str)
    .tolist()
)

if MAX_SENTENCES is not None:
    raw_texts = raw_texts[:MAX_SENTENCES]

print("Loaded sentences:", len(raw_texts))
print("Sample:")
for s in raw_texts[:5]:
    print("-", repr(s)) 

✅ Found file at: /workspace/sents.tsv
Columns found: ['id', 'sentence', 'lemmas', 'pos']
✅ Using column: 'sentence'
Loaded sentences: 35000
Sample:
- 'Балзена се смята ,'
- 'че е крепост та край село Главан ,'
- 'Гълъбовско .'
- 'На около 2 км южно от село Главан ,'
- 'върху билото на един от северните ридове на Сакар планина все още могат да се видят следи от масивни стари зидове .'


## 5) Clean and normalize Bulgarian text

In [29]:

def normalize_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)          
    text = text.replace("„", '"').replace("“", '"').replace("”", '"')
    text = text.replace("’", "'").replace("‘", "'")
    return text

def is_good_sentence(text: str) -> bool:
    if not text or len(text) < 3:
        return False
    if len(text) > 300:
        return False
    letters = sum(ch.isalpha() for ch in text)
    if letters < 3:
        return False
    return True

clean_texts = []
seen = set()

for t in raw_texts:
    t = normalize_text(t)
    if is_good_sentence(t) and t not in seen:
        seen.add(t)
        clean_texts.append(t)

print("Clean sentences:", len(clean_texts))
print("Example clean sentences:")
for s in clean_texts[:5]:
    print("-", s)

Clean sentences: 22993
Example clean sentences:
- Балзена се смята ,
- че е крепост та край село Главан ,
- Гълъбовско .
- На около 2 км южно от село Главан ,
- върху билото на един от северните ридове на Сакар планина все още могат да се видят следи от масивни стари зидове .


## 6) Synthetic noise generation

In [30]:


BG_PUNCT = [".", ",", "!", "?", ":", ";"]
BG_CHARS = "абвгдеёжзийклмнопрстуфхцчшщъьюяАБВГДЕЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЬЮЯ"

def strip_terminal_punct(text: str) -> str:
    return re.sub(r"[\.!?…]+$", "", text)

def delete_punctuation(text: str) -> str:
    text = re.sub(r"[\.,!?;:]+", "", text)
    return re.sub(r"\s+", " ", text).strip()

def random_case_noise(text: str) -> str:
    words = text.split()
    if not words:
        return text
    if random.random() < 0.5:
        words[0] = words[0].lower()
    return " ".join(words)

def char_typo(text: str) -> str:
    if len(text) < 4:
        return text
    ops = ["delete", "insert", "swap", "substitute"]
    op = random.choice(ops)
    i = random.randint(0, len(text) - 1)
    chars = list(text)

    if op == "delete":
        del chars[i]
    elif op == "insert":
        chars.insert(i, random.choice(BG_CHARS))
    elif op == "swap" and i < len(chars) - 1:
        chars[i], chars[i + 1] = chars[i + 1], chars[i]
    elif op == "substitute":
        chars[i] = random.choice(BG_CHARS)
    return "".join(chars)

def word_delete(text: str) -> str:
    words = text.split()
    if len(words) <= 2:
        return text
    i = random.randint(0, len(words) - 1)
    del words[i]
    return " ".join(words)

def word_duplicate(text: str) -> str:
    words = text.split()
    if not words:
        return text
    i = random.randint(0, len(words) - 1)
    words.insert(i, words[i])
    return " ".join(words)

def word_swap(text: str) -> str:
    words = text.split()
    if len(words) < 2:
        return text
    i = random.randint(0, len(words) - 2)
    words[i], words[i + 1] = words[i + 1], words[i]
    return " ".join(words)

def spacing_noise(text: str) -> str:
    """Fix spacing around punctuation."""
    text = re.sub(r"\s+,\s+", ", ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"([.,!?;:])\s+", r"\1 ", text)
    return text

def punctuation_noise(text: str) -> str:
    text = strip_terminal_punct(text)
    if random.random() < 0.7:
        return delete_punctuation(text)
    return text

def normalize_for_ops(text: str) -> str:
    return normalize_text(text)

def strip_punct(word: str) -> str:
    return re.sub(r"^[^\\wА-Яа-яІЇЄѝЍ]+|[^\\wА-Яа-яІЇЄѝЍ]+$", "", word)

def preserve_case(src: str, dst: str) -> str:
    if not src or not dst:
        return dst
    if src[0].isupper():
        return dst[0].upper() + dst[1:] if len(dst) > 0 else dst
    return dst

def replace_word_once(text: str, src: str, dst: str, occurrence: int = 0) -> str:
    words = text.split()
    count = 0
    for i, w in enumerate(words):
        if strip_punct(w).lower() == src.lower():
            if count == occurrence:
                words[i] = preserve_case(w, dst)
                return " ".join(words)
            count += 1
    return text

def replace_suffix_once(text: str, suffix: str, new_suffix: str, occurrence: int = 0) -> str:
    words = text.split()
    count = 0
    for i, w in enumerate(words):
        raw = strip_punct(w)
        if raw.lower().endswith(suffix.lower()) and len(raw) > len(suffix) + 1:
            if count == occurrence:
                prefix = raw[:-len(suffix)]
                new_word = prefix + new_suffix
                words[i] = preserve_case(w, new_word)
                return " ".join(words)
            count += 1
    return text

ARTICLE_SWAPS = [("ът", "а"), ("ят", "я"), ("та", "а"), ("то", "о")]

def article_suffix_for(word):
    lw = strip_punct(word).lower()
    if not lw:
        return None
    if lw.endswith(("а", "я")):
        return "та"
    if lw.endswith(("о", "е", "ю")):
        return "то"
    if lw and lw[-1] in "бвгджзйклмнпрстфхцчшщ":
        return random.choice(["ът", "ят"])
    return None

def induce_article_misuse(text: str) -> str:
    candidates = []
    for w in text.split():
        raw = strip_punct(w)
        for suf, repl in ARTICLE_SWAPS:
            if raw.lower().endswith(suf) and len(raw) > len(suf) + 2:
                candidates.append((suf, repl))
                break
    if not candidates:
        return text
    suf, repl = random.choice(candidates)
    return replace_suffix_once(text, suf, repl)

def induce_article_insertion(text: str) -> str:
    candidates = []
    for w in text.split():
        raw = strip_punct(w)
        if any(raw.lower().endswith(suf) for suf, _ in ARTICLE_SWAPS):
            continue
        suf = article_suffix_for(raw)
        if suf and len(raw) > 2:
            candidates.append((raw, suf))
    if not candidates:
        return text
    raw, suf = random.choice(candidates)
    return replace_word_once(text, raw, raw + suf)

PRONOUN_GROUPS = [
    ["той", "него"], ["тя", "нея"], ["те", "тях"],
    ["ние", "нас"], ["вие", "вас"],
]
PRONOUN_MAP = {a: b for grp in PRONOUN_GROUPS for a, b in (grp, grp[::-1])}

def induce_pronoun_misuse(text: str) -> str:
    words = text.split()
    cands = [strip_punct(w).lower() for w in words if strip_punct(w).lower() in PRONOUN_MAP]
    if not cands:
        return text
    src = random.choice(cands)
    dst = PRONOUN_MAP[src]
    return replace_word_once(text, src, dst)

VERB_SWAP = { ... }   

SPELLING_OPS = [
    punctuation_noise, random_case_noise, char_typo,
    word_delete, word_duplicate, word_swap, spacing_noise,
]

GRAMMAR_OPS = [
    induce_article_misuse, induce_article_insertion,
    induce_pronoun_misuse, induce_relative_pronoun_misuse,
    induce_demonstrative_agreement, induce_verb_agreement,
    induce_subject_verb_agreement, induce_noun_adj_disagree,
    induce_predicative_adj_agreement, induce_preposition_swap,
    induce_noun_inflection_shift, induce_word_order_swap,
]

def mixed_corrupt(text: str) -> str:
    noisy = normalize_for_ops(text)

    ops = []
    if random.random() < 0.85:
        ops.append(random.choice(GRAMMAR_OPS))
    if random.random() < 0.85:
        ops.append(random.choice(SPELLING_OPS))
    if random.random() < 0.45:
        ops.append(random.choice([punctuation_noise, spacing_noise, random_case_noise]))

    if not ops:
        ops.append(random.choice(GRAMMAR_OPS + SPELLING_OPS))

    random.shuffle(ops)
    for fn in ops:
        try:
            candidate = fn(noisy)
            if candidate and candidate != noisy:
                noisy = candidate
        except Exception:
            pass

    noisy = normalize_text(noisy)
    if not noisy:
        noisy = text
    return noisy

print("=== Noise Examples ===")
for _ in range(8):
    s = random.choice(clean_texts)
    print("CLEAN :", s)
    print("NOISY :", mixed_corrupt(s))
    print("-" * 80)

=== Noise Examples ===
CLEAN : акули и дори други млади или дребни китове .
NOISY : акули и дори млади други или дребни китове .
--------------------------------------------------------------------------------
CLEAN : == Обща информация == Ръжта е едногодишно тревисто растение .
NOISY : Обща == информация == Ръжта е едногодишно тревисто растение.
--------------------------------------------------------------------------------
CLEAN : contains 100 &
NOISY : 100 &
--------------------------------------------------------------------------------
CLEAN : но може да останат за известно време ,
NOISY : но може да останат за известно време ,
--------------------------------------------------------------------------------
CLEAN : Именно в този щат е открит и описан за науката видът <
NOISY : Именно в този щат е открит и описан за науката видът <
--------------------------------------------------------------------------------
CLEAN : стъбла (
NOISY : стъбла (
------------------------------------

## 7) Build correction pairs

In [31]:

pairs = []
for sent in clean_texts:
    if random.random() < IDENTITY_RATIO:
        pairs.append({
            "input_text": f"correct_bg: {sent}",
            "target_text": sent
        })

    for _ in range(NOISY_VARIANTS_PER_SENTENCE):
        noisy = mixed_corrupt(sent)
        if noisy != sent:
            pairs.append({
                "input_text": f"correct_bg: {noisy}",
                "target_text": sent
            })

pairs_df = pd.DataFrame(pairs)
pairs_df = pairs_df.drop_duplicates(subset=["input_text", "target_text"]).reset_index(drop=True)

print("Total pairs:", len(pairs_df))
print(pairs_df.sample(5, random_state=42))


Total pairs: 73056
                                              input_text  \
11987  correct_bg: ЮАР и държавите от социалистически...   
30999    correct_bg: но повечето събират в големи групи,   
25356                                 correct_bg: с клюн   
24327                               correct_bg: Канзасят   
10240                                  correct_bg: матка   

                                      target_text  
11987  ЮАР и държавите от социалистическия блок .  
30999     но повечето се събират в големи групи ,  
25356                           Почуквайки с клюн  
24327                                    Канзас ,  
10240                                     матка ,  


## 8) Train / validation / test split

In [32]:

train_df, test_df = train_test_split(
    pairs_df,
    test_size=TEST_SIZE,
    random_state=42,
    shuffle=True,
)

train_df, val_df = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=42,
    shuffle=True,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset

Train: 59175
Validation: 6575
Test: 7306


DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 59175
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 6575
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7306
    })
})

## 9) Tokenizer and preprocessing

In [33]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def preprocess_batch(batch):
    inputs = batch["input_text"]
    targets = batch["target_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding="longest",
)

tokenized

Map:   0%|          | 0/59175 [00:00<?, ? examples/s]

Map:   0%|          | 0/6575 [00:00<?, ? examples/s]

Map:   0%|          | 0/7306 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 59175
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6575
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7306
    })
})

## 10) Load the mT5 model

In [34]:


model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Model loaded:", MODEL_NAME)
print("Num parameters:", round(sum(p.numel() for p in model.parameters())/1e6, 2), "M")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded: google/mt5-base
Num parameters: 582.4 M


## 11) Metrics

In [38]:


def normalize_for_eval(text: str) -> str:
    text = normalize_text(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def simple_token_f1(pred: str, ref: str) -> float:
    pred_tokens = normalize_for_eval(pred).split()
    ref_tokens = normalize_for_eval(ref).split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    ref_counts = {}
    for tok in ref_tokens:
        ref_counts[tok] = ref_counts.get(tok, 0) + 1
    common = 0
    for tok in pred_tokens:
        if ref_counts.get(tok, 0) > 0:
            common += 1
            ref_counts[tok] -= 1
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    if preds is not None:
        preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
        preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    exact = 0
    f1_scores = []
    for p, r in zip(decoded_preds, decoded_labels):
        p_norm = normalize_for_eval(p)
        r_norm = normalize_for_eval(r)
        if p_norm == r_norm:
            exact += 1
        f1_scores.append(simple_token_f1(p, r))

    return {
        "exact_match": exact / len(decoded_preds),
        "token_f1": float(np.mean(f1_scores)),
    }

## 12) Training setup

In [40]:

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=6,         
    gradient_accumulation_steps=2,
    num_train_epochs=3,                     
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,                 
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="token_f1",
    greater_is_better=True,
    fp16=USE_FP16,
    bf16=USE_BF16,
    report_to="none",
    dataloader_num_workers=0,
    optim="adamw_torch",
    warmup_steps=0.1,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer


## 13) Train

In [41]:


train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss,Exact Match,Token F1
1,0.462140,0.228375,0.643498,0.938019
2,0.310934,0.198213,0.688365,0.944157
3,0.201325,0.200895,0.702662,0.947202


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=22191, training_loss=0.34421824811572965, metrics={'train_runtime': 5872.452, 'train_samples_per_second': 30.23, 'train_steps_per_second': 3.779, 'total_flos': 1.1712102289265664e+16, 'train_loss': 0.34421824811572965, 'epoch': 3.0})

## 14) Evaluate on the test set

In [42]:


test_metrics = trainer.evaluate(tokenized["test"])
print(test_metrics)

Training Loss,Validation Loss,Epoch,Exact Match,Token F1
0.201325,0.192867,3,0.708733,0.948979


{'eval_loss': 0.1928674578666687, 'eval_exact_match': 0.7087325485901999, 'eval_token_f1': 0.948978585795094}


## 15) Save the model

In [43]:


os.makedirs(OUTPUT_DIR, exist_ok=True)
model.config.tie_word_embeddings = False
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./mt5_bg_full_corrector


## 16) Inference helper

In [44]:


device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def correct_bg_text(text: str, max_new_tokens: int = 128) -> str:
    prompt = f"correct_bg: {normalize_text(text)}"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

examples = [
    "аз обича да чета книги",
    "Днес времето е прекрасно",
    "той отива на училище без да яде закуска",
    "Голямата планина, в който се качихме, беше голяма",
    "Те отиде на кино вчера",
    "Жената е красив.",
    "Той каза че ще дойде",
    "На маса лежеше книга",
]

for ex in examples:
    print("INPUT   :", ex)
    print("OUTPUT  :", correct_bg_text(ex))
    print()


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INPUT   : аз обича да чета книги
OUTPUT  : аз обича да чета книга ,

INPUT   : Днес времето е прекрасно


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OUTPUT  : Днес времето е прекрасо ,

INPUT   : той отива на училище без да яде закуска
OUTPUT  : той отива на училище без да яде закуска .

INPUT   : Голямата планина, в който се качихме, беше голяма


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OUTPUT  : Голямата планина в който се качихме ,

INPUT   : Те отиде на кино вчера
OUTPUT  : Те отиде на кино вчера ,

INPUT   : Жената е красив.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OUTPUT  : Жената е красива .

INPUT   : Той каза че ще дойде
OUTPUT  : Той каза че ще дойде ,

INPUT   : На маса лежеше книга
OUTPUT  : На маса лежеше книга ,



In [47]:

import shutil
import os

final_model_dir = OUTPUT_DIR  


checkpoint_dirs = [d for d in os.listdir(final_model_dir) if d.startswith("checkpoint-")]
for cp in checkpoint_dirs:
    cp_path = os.path.join(final_model_dir, cp)
    print(f"Removing old checkpoint: {cp}")
    shutil.rmtree(cp_path, ignore_errors=True)

for f in ["trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth"]:
    path = os.path.join(final_model_dir, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed: {f}")

print("Cleaned model folder. Current files:")
for item in os.listdir(final_model_dir):
    size = os.path.getsize(os.path.join(final_model_dir, item)) / (1024*1024)
    print(f" - {item} ({size:.2f} MB)")

Removing old checkpoint: checkpoint-14794
Removing old checkpoint: checkpoint-22191
✅ Cleaned model folder. Current files:
 - config.json (0.00 MB)
 - generation_config.json (0.00 MB)
 - model.safetensors (2221.72 MB)
 - tokenizer_config.json (0.00 MB)
 - tokenizer.json (15.57 MB)
 - training_args.bin (0.01 MB)


In [48]:

import shutil
import os

model_path = OUTPUT_DIR

zip_filename = "mt5_bg_full_corrector.zip"

print(f"Creating zip file: {zip_filename} ...")
shutil.make_archive(
    base_name=zip_filename.replace(".zip", ""), 
    format="zip", 
    root_dir=model_path
)

print(f"Model successfully zipped: {zip_filename}")
print(f"Size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")

from IPython.display import FileLink
display(FileLink(zip_filename))

Creating zip file: mt5_bg_full_corrector.zip ...
✅ Model successfully zipped: mt5_bg_full_corrector.zip
Size: 1783.46 MB


/workspace/mt5_bg_full_corrector.zip